In [6]:
import pandas as pd
import altair as alt
import os

df = pd.read_csv('results.csv')
df['resolution'] = df['screen_pct'].astype(str) + '%'

metrics = [
    ('avg_GPU/TAA',            'TAA GPU Cost (ms)'),
    ('avg_GPUTime',            'Total GPU Time (ms)'),
    ('avg_RenderThreadTime',   'Render Thread Time (ms)'),
    ('avg_GPUMem/LocalUsedMB', 'VRAM (MB)'),
]

taa_params = [
    'r.TemporalAACurrentFrameWeight',
    'r.TemporalAASamples',
    'r.TemporalAAFilterSize',
    'r.TemporalAA.HistoryScreenPercentage',
]

res_order  = ['100%', '87%', '71%', '50%']
res_colors = ['#2a7db5', '#2ca05a', '#e07b3d', '#d4504a']
color_scale = alt.Scale(domain=res_order, range=res_colors)

os.makedirs('plots', exist_ok=True)

for scene in df['scene'].unique():
    scene_df = df[df['scene'] == scene]
    
    rows = []
    for metric_col, metric_label in metrics:
        cols = []
        for param in taa_params:
            subset = scene_df[scene_df['cvar_name'] == param].copy()

            if subset.empty:
                continue

            # short param label for column title
            short_param = param.split('r.')[-1]

            chart = alt.Chart(subset).mark_line(point=True).encode(
                x=alt.X('cvar_value:Q', title=short_param),
                y=alt.Y(f'{metric_col}:Q', title=metric_label),
                color=alt.Color('resolution:N', scale=color_scale, sort=res_order,
                                legend=alt.Legend(title='Resolution')),
                tooltip=['resolution', 'cvar_value', metric_col]
            ).properties(
                width=220,
                height=180
            )
            cols.append(chart)

        if cols:
            rows.append(alt.hconcat(*cols))

    if rows:
        full_grid = alt.vconcat(*rows).properties(
            title=alt.TitleParams(
                text=scene,
                fontSize=16,
                anchor='start'
            )
        ).resolve_scale(
            y='independent'  # each row has its own y scale
        )

        full_grid.save(f'plots/{scene}_grid.html')
        full_grid.display()

alt.VConcatChart(...)

alt.VConcatChart(...)

In [5]:
df = pd.read_csv('results.csv')
subset = df[df['cvar_name'] == 'r.TemporalAA.HistoryScreenPercentage']
print(subset[['scene', 'screen_pct', 'cvar_value', 'avg_GPUTime', 'avg_GPU/TAA']])

              scene  screen_pct  cvar_value  avg_GPUTime  avg_GPU/TAA
24         cubetest         100       100.0       9.9974       0.1765
25         cubetest         100       125.0      10.9828       0.2001
26         cubetest         100       150.0      10.7584       0.1947
27         cubetest         100       200.0      10.5095       0.2005
52         cubetest          87       100.0      13.3833       0.3102
53         cubetest          87       125.0      13.4227       0.2862
54         cubetest          87       150.0      10.0438       0.1998
55         cubetest          87       200.0      13.4008       0.3067
80         cubetest          71       100.0       7.4262       0.1223
81         cubetest          71       125.0       9.1274       0.1781
82         cubetest          71       150.0      10.7007       0.2071
83         cubetest          71       200.0       9.2959       0.1778
108        cubetest          50       100.0      10.2019       0.1814
109        cubetest 

In [3]:
df = pd.read_csv('results.csv')

# print all rows for one specific parameter to see if values actually vary
subset = df[df['cvar_name'] == 'r.TemporalAA.HistoryScreenPercentage']
print(subset[['scene', 'screen_pct', 'cvar_value', 'avg_GPUTime', 'avg_GPUMem/LocalUsedMB', 'frames_analysed']])

        scene  screen_pct  cvar_value  avg_GPUTime  avg_GPUMem/LocalUsedMB  \
24   cubetest         100       100.0       9.9974               2622.7804   
25   cubetest         100       125.0      10.9828               2623.8086   
26   cubetest         100       150.0      10.7584               2624.0154   
27   cubetest         100       200.0      10.5095               2623.7804   
52   cubetest          87       100.0      13.3833               2561.1211   
53   cubetest          87       125.0      13.4227               2560.3429   
54   cubetest          87       150.0      10.0438               2561.1383   
55   cubetest          87       200.0      13.4008               2561.0219   
80   cubetest          71       100.0       7.4262               2478.0319   
81   cubetest          71       125.0       9.1274               2478.0617   
82   cubetest          71       150.0      10.7007               2479.0617   
83   cubetest          71       200.0       9.2959              